- Henter, renser, splitter og gemmer Adult datasættet i CSV filer i data/
- Den skal køres én gang før de andre notebooks som indeholder vores modeller.

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

- UCI fetch kan nogle gange være langsom, og kræver internet.
- Vi gemmer derfor den rå data lokalt: data/adult_raw.csv, så de efterfølgende runs er instant.

In [2]:
#definerer sti og sikrer at data mappen eksisterer
PATH = Path('data/adult_raw.csv')
PATH.parent.mkdir(exist_ok=True)

#hvis der er cache hit henter vi den lokalt og undgår et kald
if PATH.exists():
    data = pd.read_csv(PATH)

#hvis cache er miss henter vi det fra UCI én gang og gemmer lokalt
else:
    adult = fetch_ucirepo(id=2)
    data = pd.concat([adult.data.features, adult.data.targets], axis=1)
    data.to_csv(PATH, index=False)

#splitter det i features og target
X = data.drop(columns=['income'])
y = data['income']

- Grunden til at vi renser target er at income har 4 unikke labels: '<=50K', '<=50K.', '>50K', '>50K.'
- pga. trailing '.', Der er reelt kun 2 klasser, så vi stripper og fjerner '.', så vi får binær target.

In [3]:
y_clean = y.str.strip().str.rstrip('.')
y_clean.value_counts()

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

- Vi laver '?' til 'NaN' så det ikke bliver sin egen one-hot kategori.
- Vi dropper education for eduction-num repræsenterer det samme information numerisk og hierarkisk.
- dropna() smider ca. 3K rækker (~48k til ~45k), det er samme balance og vi skal ikke imputere.
- I native-country dominerer USA, og vi grupperer alt udenfor top 9 i 'Other' så vi har 10 kategorier istedet for 42.

In [4]:
#feature cleaning
X_clean = X.replace('?', np.nan)
X_clean = X_clean.drop(columns='education')
X_clean = X_clean.dropna()
y_clean = y_clean.loc[X_clean.index] #sørger for at X og y er i sync efter dropna
top9 = X_clean['native-country'].value_counts().head(9).index
X_clean['native-country'] = X_clean['native-country'].where(X_clean['native-country'].isin(top9), 'Other')

- Datasættet er det der bliver beskrevet som moderat ubalanceret (~75/25)
- Stratify sikrer at fordeling af det vi vælger at stratify på, er ens i train/test/val, så vi undgår sampling bias - vi gør det på target
- Vi laver tre sæt: train(fit), val(fine-tuning af hp), test.
- Random state = 42 gør at vi kan reproducerer sættet igen, hvilket er vigtigt hvis vi skal sammenligne de to modeller på de samme rows
- Vi fordeler sættet 80/20 - train+val & test
- Herefter deles train+val igen i 80/20
- Reelt set er det: 64/16/20%

In [5]:
# To stratified splits: først train_full/test, derefter train/val ud af train_full
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_clean, y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

# Sanity check: klassefordelingen skal være ens på tværs
print('Shapes:')
print(f'  X_train: {X_train.shape}')
print(f'  X_val  : {X_val.shape}')
print(f'  X_test : {X_test.shape}')
print(f'  total  : {len(X_train) + len(X_val) + len(X_test)} (forventer {len(X_clean)})')

print('\nKlassefordeling (skal være ~ens pga. stratify):')
print('train:', y_train.value_counts(normalize=True).round(3).to_dict())
print('val  :', y_val.value_counts(normalize=True).round(3).to_dict())
print('test :', y_test.value_counts(normalize=True).round(3).to_dict())

Shapes:
  X_train: (28941, 13)
  X_val  : (7236, 13)
  X_test : (9045, 13)
  total  : 45222 (forventer 45222)

Klassefordeling (skal være ~ens pga. stratify):
train: {'<=50K': 0.752, '>50K': 0.248}
val  : {'<=50K': 0.752, '>50K': 0.248}
test : {'<=50K': 0.752, '>50K': 0.248}


- Vi gemmer den splittede data rå inden vi encoder, fordi at MLP og HGBT kræver forskellig encoding.
- Hvis vi encoder her, kan vi ikke genbruge filerne.
- På den her måde kan hver notebook stå for sin egen encoding pipeline.

In [6]:
OUT = Path('data')
OUT.mkdir(exist_ok=True)

# X_* er DataFrames > index=False så vi ikke får en ekstra unavngiven kolonne ved reload
X_train.to_csv(OUT / 'X_train.csv', index=False)
X_val.to_csv(OUT / 'X_val.csv', index=False)
X_test.to_csv(OUT / 'X_test.csv', index=False)

# y_* er Series > header=True bevarer kolonnenavnet 'income'
y_train.to_csv(OUT / 'y_train.csv', index=False, header=True)
y_val.to_csv(OUT / 'y_val.csv', index=False, header=True)
y_test.to_csv(OUT / 'y_test.csv', index=False, header=True)